# Phase 22 — causal and out-of-sample validation

All selection rules in this notebook are written before model outputs are inspected. The history and symbols have already been explored, so these results remain **exploratory chronological OOS evidence**, not a pristine final test.

In [1]:
import os

# Reproducible HMM fits (M15): fix the BLAS/OpenMP and PyTorch thread counts before the numerical
# libraries load. A concurrent or differently threaded run can change floating-point reduction order
# and therefore the EM path. Restart the kernel after changing HMM_THREADS.
HMM_THREADS = 1
for _thread_variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_thread_variable, str(HMM_THREADS))

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from dc_basics.data import resolve_data_root
from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    UNMATCHED_REGIME_CODE,
    add_future_diagnostics,
    build_multiscale_state,
    dwell_lengths,
    emission_feature_contributions,
    incremental_usefulness,
    load_close,
    permutation_omega_test,
    prepare_data,
    ratio_ladder,
    select_hmm_candidates,
)

torch.set_num_threads(HMM_THREADS)

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
STABILITY_SEEDS = (29, 47)
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
MINIMUM_RESTART_ARI = 0.80
MIN_EMISSION_VARIANCE = 1e-2
N_PERMUTATIONS = 499
FUTURE_HORIZONS = (6, 24, 96)
N_BOOTSTRAP = 499
BOOTSTRAP_BLOCK_ROWS = 20
BOOTSTRAP_CONFIDENCE = 0.90

## Frozen selection rule

A candidate is eligible only when:

1. fitting and causal filtering complete without error;
2. the filtering prefix-invariance audit passes;
3. every validation state occupies at least 2% of observations;
4. every declared restart completes and the minimum validation adjusted-Rand agreement with the primary fit is at least 0.80.

Among eligible candidates, choose the highest validation log-likelihood per observation. State count, emission family, and random seed are never selected on test results. The test partition is opened only after selection. Variance-floor diagnostics remain visible; high likelihood or confidence is not interpreted in isolation.

Before any fit, `prepare_data` screens the emission features on training rows only: training-constant columns, columns with fewer than three distinct values, and zero-inflated columns whose most frequent value covers at least 20% of rows are removed. The removed columns and their reasons are printed below.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, CLOCK, FEATURE_TIER)
print(f"Train: {len(prepared.frames['train']):,}")
print(f"Validation: {len(prepared.frames['validation']):,}")
print(f"Test: {len(prepared.frames['test']):,}")
print(f"Retained features: {len(prepared.feature_columns)}")
print(f"Removed emission features and reasons: {prepared.removed_feature_reasons}")
print(f"Dropped rows by structural regime: {prepared.dropped_regime_counts}")

Train: 3,273
Validation: 938
Test: 851
Retained features: 21
Removed training-constant features: ('reversal_progress_00',)
Dropped rows by structural regime: {'broad_coherence': 0, 'shallow_counter': 0, 'deep_counter': 0, 'fragmented': 0}


In [3]:
selection = select_hmm_candidates(
    prepared,
    HMM_FAMILIES,
    STATE_COUNTS,
    random_state=RANDOM_STATE,
    stability_seeds=STABILITY_SEEDS,
    max_iter=MAX_ITER,
    n_restarts=N_RESTARTS,
    minimum_validation_occupancy=MINIMUM_VALIDATION_OCCUPANCY,
    minimum_restart_ari=MINIMUM_RESTART_ARI,
    min_emission_variance=MIN_EMISSION_VARIANCE,
)
selection_table = selection.table
display(selection_table.set_index(["family", "n_states"]))
selected_fit = selection.selected_fit
if selected_fit is None:
    raise RuntimeError("No candidate passed the frozen validation eligibility rules.")
print(f"Selected on validation only: {selected_fit.family.value}, K={selected_fit.n_states}")

clock     tier  feature_count  \
family       n_states                                             
normal_hmm   3         union_event_time  dynamic             21   
             4         union_event_time  dynamic             21   
             5         union_event_time  dynamic             21   
studentt_hmm 3         union_event_time  dynamic             21   
             4         union_event_time  dynamic             21   
             5         union_event_time  dynamic             21   

                       removed_feature_count  prepared_rows_dropped error  \
family       n_states                                                       
normal_hmm   3                             1                      0  None   
             4                             1                      0  None   
             5                             1                      0  None   
studentt_hmm 3                             1                      0  None   
             4                             1                      0  None   
             5                             1                      0  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -22.590537   
             4                                -19.738444   
             5                                -18.732857   
studentt_hmm 3                                -22.896883   
             4                                -21.071885   
             5                                -20.080213   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.292111     0.947705           1.0   
             4                        0.156716     0.896478           1.0   
             5                        0.126866     0.935966           1.0   
studentt_hmm 3                        0.239872     0.788687           1.0   
             4                        0.124733     0.872999           1.0   
             5                        0.131130     0.899680           1.0   

                       ...  baseline_nmi  semantic_mapping_coverage  \
family       n_states  ...                                            
normal_hmm   3         ...      0.330369                   1.000000   
             4         ...      0.511658                   1.000000   
             5         ...      0.266190                   0.786780   
studentt_hmm 3         ...      0.201646                   1.000000   
             4         ...      0.387654                   1.000000   
             5         ...      0.439902                   0.760128   

                       baseline_mapped_agreement  minimum_fitted_variance  \
family       n_states                                                       
normal_hmm   3                          0.479744                     0.01   
             4                          0.651386                     0.01   
             5                          0.510840                     0.01   
studentt_hmm 3                          0.473348                     0.01   
             4                          0.603412                     0.01   
             5                          0.807854                     0.01   

                       variance_floor_hit_count  variance_floor_hit_share  \
family       n_states                                                       
normal_hmm   3                                3                  0.047619   
             4                                5                  0.059524   
             5                               13                  0.123810   
studentt_hmm 3                                5                  0.079365   
             4                                5                  0.059524   
             5                                6          

Selected on validation only: normal_hmm, K=4


## Restart stability

Adjusted Rand agreement is permutation-invariant. It tests whether each family/state-count repeatedly partitions validation observations similarly, not whether its semantic names are correct. Stability is an eligibility requirement evaluated for **every** candidate before likelihood ranking.

In [4]:
selected_stability = selection.stability.loc[
    selection.stability["family"].eq(selected_fit.family.value)
    & selection.stability["n_states"].eq(selected_fit.n_states)
]
display(selection.stability.set_index(["family", "n_states", "seed"]))
print("Selected-candidate restart audit:")
display(selected_stability.set_index("seed"))

error  validation_ari_vs_primary  \
family       n_states seed                                    
normal_hmm   3        29    None                   0.769205   
                      47    None                   0.668122   
             4        29    None                   1.000000   
                      47    None                   1.000000   
             5        29    None                   0.545847   
                      47    None                   0.783602   
studentt_hmm 3        29    None                   1.000000   
                      47    None                   0.724244   
             4        29    None                   1.000000   
                      47    None                   1.000000   
             5        29    None                   0.664581   
                      47    None                   0.757572   

                            validation_log_likelihood_per_row  \
family       n_states seed                                      
normal_hmm   3        29                           -21.578788   
                      47                           -22.143909   
             4        29                           -19.738497   
                      47                           -19.738903   
             5        29                           -18.453852   
                      47                           -18.806643   
studentt_hmm 3        29                           -22.894412   
                      47                           -23.661815   
             4        29                           -21.071653   
                      47                           -21.071653   
             5        29                           -20.063935   
                      47                           -20.319191   

                            prefix_invariant  
family       n_states seed                    
normal_hmm   3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True  
studentt_hmm 3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True

Selected-candidate restart audit:


,family,n_states,error,validation_ari_vs_primary,validation_log_likelihood_per_row,prefix_invariant
seed,,,,,,
29,normal_hmm,4,None,1.0,-19.738497,True
47,normal_hmm,4,None,1.0,-19.738903,True


## Where the validation likelihood comes from

Likelihood ranks candidates, so it must be visible what supplies it. With diagonal covariances each candidate's validation log density splits exactly by feature. The first table decomposes the selected model; the second lists, for every candidate, the features where at least one state sits at the variance floor and the share of the likelihood gain they supply. A gain concentrated in floored features is a numerical artefact, not a better description of the data.

In [ ]:
selected_contributions = emission_feature_contributions(selected_fit, prepared, "validation")
display(selected_contributions.style.format({
    "weighted_log_density": "{:.3f}",
    "gain_vs_standard_normal": "{:.3f}",
    "training_point_mass_share": "{:.2%}",
    "share_of_total_gain": "{:.1%}",
}))

floor_records = []
for candidate in selection.fits:
    contributions = emission_feature_contributions(candidate, prepared, "validation")
    if contributions.empty:
        continue
    floored = contributions.loc[contributions["states_at_variance_floor"].gt(0)]
    total_gain = contributions["gain_vs_standard_normal"].sum()
    floor_records.append({
        "family": candidate.family.value,
        "n_states": candidate.n_states,
        "features_with_floored_state": len(floored),
        "floored_features": tuple(floored.index),
        "floored_share_of_gain": floored["gain_vs_standard_normal"].sum() / total_gain if total_gain else np.nan,
    })
display(pd.DataFrame.from_records(floor_records).set_index(["family", "n_states"]).style.format({
    "floored_share_of_gain": "{:.1%}",
}))

## Open the chronological test partition

These metrics are reported after validation selection. They evaluate stability, persistence, confidence, and relationship to the direct baseline. No test metric is used to repair or reselect the model.

In [5]:
test_states = selected_fit.states["test"]
test_probabilities = selected_fit.probabilities["test"]
test_baseline = prepared.frames["test"]["structural_regime_code"].to_numpy(dtype=np.int64)
test_occupancy = np.bincount(test_states, minlength=selected_fit.n_states) / len(test_states)
test_dwell = dwell_lengths(test_states)
mapped_test = selected_fit.mapped_regimes["test"]
mapped_mask = mapped_test != UNMATCHED_REGIME_CODE
test_metrics = pd.Series({
    "log_likelihood_per_row": selected_fit.log_likelihoods["test"] / len(test_states),
    "minimum_state_occupancy": test_occupancy.min(),
    "switch_rate": np.mean(test_states[1:] != test_states[:-1]),
    "median_dwell": np.median(test_dwell),
    "p90_dwell": np.quantile(test_dwell, 0.90),
    "mean_max_filtered_probability": test_probabilities.max(axis=1).mean(),
    "baseline_adjusted_rand": adjusted_rand_score(test_baseline, test_states),
    "baseline_nmi": normalized_mutual_info_score(test_baseline, test_states),
    "semantic_mapping_coverage": mapped_mask.mean(),
    "baseline_mapped_agreement": (
        np.mean(test_baseline[mapped_mask] == mapped_test[mapped_mask]) if mapped_mask.any() else np.nan
    ),
})
display(test_metrics.to_frame("test_value"))

,test_value
log_likelihood_per_row,-22.251451
minimum_state_occupancy,0.144536
switch_rate,0.880000
median_dwell,1.000000
p90_dwell,1.000000
mean_max_filtered_probability,0.997735
baseline_adjusted_rand,0.344083
baseline_nmi,0.475574
semantic_mapping_coverage,1.000000
baseline_mapped_agreement,0.649824


## Diagnostic usefulness beyond renaming the baseline

Forward activity and absolute movement are diagnostic outcomes, not trading targets. Bias-corrected omega-squared measures held-out separation, and a label-permutation null checks whether that separation exceeds what group count and sample noise create mechanically.

Each model partition is compared with both the direction-neutral structural taxonomy and the direction-aware direct taxonomy. This does not establish causation, profitability, or directional predictability.

The permutation null shuffles rows independently, so it ignores the serial dependence created by overlapping forward windows; treat its p-values as optimistic. The last table therefore compares the model with the stronger direct baseline for each diagnostic and puts a moving-block bootstrap interval on the difference (contiguous blocks of `BOOTSTRAP_BLOCK_ROWS` events, `BOOTSTRAP_CONFIDENCE` coverage). The `verdict` column is computed from that interval: `adds_beyond_baselines` only when the whole interval is above zero, `baseline_separates_more` when it is below zero, and `no_information_beyond_baseline` when it contains zero.

In [6]:
diagnostic_state = add_future_diagnostics(global_state, FUTURE_HORIZONS)
test_frame = prepared.frames["test"].copy()
test_frame["model_state"] = test_states
diagnostic_columns = []
for horizon in FUTURE_HORIZONS:
    diagnostic_columns.extend([
        f"future_abs_return_{horizon}",
        f"future_realized_vol_{horizon}",
        f"future_dcc_count_{horizon}",
    ])
test_frame = test_frame.join(diagnostic_state[diagnostic_columns], how="left")
partition_columns = {
    "direct_structural": "structural_regime",
    "direct_direction_aware": "directed_regime",
    "model_state": "model_state",
}
usefulness_records = []
for diagnostic_index, column in enumerate(diagnostic_columns):
    for partition_name, label_column in partition_columns.items():
        evidence = permutation_omega_test(
            test_frame[column], test_frame[label_column], n_permutations=N_PERMUTATIONS,
            random_state=RANDOM_STATE + diagnostic_index,
        )
        usefulness_records.append({
            "diagnostic": column,
            "partition": partition_name,
            **evidence,
        })
usefulness = pd.DataFrame.from_records(usefulness_records).set_index(["diagnostic", "partition"])
display(usefulness.style.format({
    "omega_squared": "{:.4f}", "null_p95": "{:.4f}", "permutation_p_value": "{:.3f}",
}))

effect_comparison = usefulness["omega_squared"].unstack("partition")
effect_comparison["model_minus_structural"] = (
    effect_comparison["model_state"] - effect_comparison["direct_structural"]
)
effect_comparison["model_minus_direction_aware"] = (
    effect_comparison["model_state"] - effect_comparison["direct_direction_aware"]
)
display(effect_comparison.style.format("{:.4f}"))

incremental = incremental_usefulness(
    test_frame,
    diagnostic_columns,
    "model_state",
    {"direct_structural": "structural_regime", "direct_direction_aware": "directed_regime"},
    n_bootstrap=N_BOOTSTRAP,
    block_length=BOOTSTRAP_BLOCK_ROWS,
    confidence=BOOTSTRAP_CONFIDENCE,
    random_state=RANDOM_STATE,
)
numeric_columns = [column for column in incremental.columns if column not in ("best_baseline", "verdict")]
display(incremental.style.format({column: "{:.4f}" for column in numeric_columns}))
print("Verdict counts across diagnostics:")
print(incremental["verdict"].value_counts().to_string())

partition,direct_direction_aware,direct_structural,model_state,model_minus_structural,model_minus_direction_aware
diagnostic,,,,,
future_abs_return_24,0.0059,0.0048,0.0091,0.0043,0.0032
future_abs_return_6,0.0112,0.0000,0.0115,0.0115,0.0003
future_abs_return_96,0.0043,0.0039,0.0000,-0.0039,-0.0043
future_dcc_count_24,0.0184,0.0164,0.0041,-0.0123,-0.0143
future_dcc_count_6,0.0049,0.0050,0.0025,-0.0025,-0.0024
future_dcc_count_96,0.0131,0.0075,0.0013,-0.0062,-0.0118
future_realized_vol_24,0.0079,0.0065,0.0088,0.0023,0.0009
future_realized_vol_6,0.0097,0.0088,0.0349,0.0261,0.0252
future_realized_vol_96,0.0064,0.0002,0.0002,0.0000,-0.0062


# Phase 22 completion gate

1. Selection uses validation likelihood and eligibility rules, never test results.
2. Train-only scaling and train-only semantic mapping are preserved.
3. Filtered prefix invariance passes across the train/validation boundary.
4. No state collapses below the predeclared occupancy floor on validation.
5. Every candidate must clear the predeclared restart-stability threshold before likelihood ranking.
6. Unsupported semantic states remain unmatched; ARI/NMI are computed on raw state partitions.
7. Test persistence is evaluated together with confidence and switching; stickiness alone is not success.
8. Held-out diagnostic separation uses bias-corrected effect size, a permutation null, and a moving-block bootstrap interval for the model's margin over the stronger direct baseline; the verdict is computed, not asserted.
9. The model is compared with both direction-neutral and direction-aware direct baselines.
10. No directional alpha or trading claim is made.
11. Because this history was previously explored, the result remains exploratory OOS evidence.
12. Per-feature likelihood contributions and variance-floor hits are inspected before likelihood differences between candidates are trusted.